# 01. Raw — Bronze Layer

This notebook implements Stage 1 of the HDB ETL pipeline: Raw ingestion.

Responsibilities:
- Inherit all configs, libraries, and the Spark session from 00. configs.
- Load all source CSV files into a single merged DataFrame.
- Apply a date range filter to restrict data to the defined scope.
- Persist the result to the Bronze layer.

No cleaning, type conversion, or business logic is applied at this stage. The Bronze layer reflects the source data as faithfully as possible within the defined date scope.

## Inherit Configs

Runs 00. configs to import all libraries, initialise the Spark session, and make all pipeline constants available in this notebook. This includes SOURCE_PATH, DATE_START, DATE_END, HDB_LEASE, VALID_FLAT_TYPES, all schema names, table names, fully qualified names.

In [0]:
%run "./00_nb_configs"

# 00. Configs

This notebook is the single source of truth for all pipeline-wide settings.
It is referenced by all downstream notebooks via %run 00. configs.

Contents:
- Required library imports
- SparkSession initialisation
- Schema and table name constants following the Medallion architecture (Bronze, Silver, Gold)
- Source data path
- Business logic constants (HDB lease duration and valid flat types)
- Date range filter boundaries
- Delta table creation

To customise the pipeline, only edit values in this notebook. All other notebooks inherit these values automatically.

## Required Libraries

## Source Data Path
Define where the raw HDB CSV files are located.


Source path : dbfs:/FileStore/test/


## Databricks - Catalog
- Databricks-specific catalog usage 
- If using a different cloud platform, comment out the below cell.

...

## Date Range Filter

The pipeline scope covers HDB resale transactions from January 2012 to December 2016 as per the requierment.

Date filter : 2012-01-01  to  2016-12-31


## Business Logic Constants

- HDB_LEASE is set to 99, which is the standard Singapore HDB lease duration in years. It is used in the Silver stage to compute the lease expiry date and the recomputed remaining lease string.

- VALID_FLAT_TYPES is the whitelist of officially recognised HDB flat types. Any record whose flat_type falls outside this list will be flagged as invalid during the validation step in the Silver stage.

HDB lease   : 99 years
Valid types : ['1 ROOM', '2 ROOM', '3 ROOM', '4 ROOM', '5 ROOM', 'EXECUTIVE', 'MULTI-GENERATION']


## Output Delta Table Creation
- Databricks-specific schema and table creation function. 
- If using a different cloud platform, comment out the schema creation line.

## Initialise Spark Session and Load Raw Data

- Read all CSV files from SOURCE_PATH into a single raw DataFrame using three key options.

- header=true treats the first row of each CSV as column names.

- mergeSchema=true unions the schemas across all CSV files so that columns present in only some files, such as remaining_lease which was added in later datasets, are included for all rows with null where absent.

- inferSchema=true automatically detects column data types from the values in the data.

In [0]:
raw_hdb_df = (
    spark.read
    .option("header", "true")
    .option("mergeSchema", "true")
    .option("inferSchema", "true")
    .csv(SOURCE_PATH)
)

print(f"Raw records loaded : {raw_hdb_df.count():,}")

Raw records loaded : 972,674


## Inspect Raw Schema

Print the inferred schema immediately after ingestion to confirm all expected columns are present across the merged files and to observe the raw types as Spark inferred them from the source data. This is useful for understanding what type corrections may be needed in the Silver stage downstream.

In [0]:
raw_hdb_df.printSchema()

root
 |-- month: date (nullable = true)
 |-- town: string (nullable = true)
 |-- flat_type: string (nullable = true)
 |-- block: string (nullable = true)
 |-- street_name: string (nullable = true)
 |-- storey_range: string (nullable = true)
 |-- floor_area_sqm: double (nullable = true)
 |-- flat_model: string (nullable = true)
 |-- lease_commence_date: integer (nullable = true)
 |-- resale_price: double (nullable = true)
 |-- remaining_lease: string (nullable = true)



## Filter Records to Target Date Range

Restrict the dataset to the scope defined in configs: DATE_START to DATE_END, which covers January 2012 to December 2016 inclusive.

The month column representing the transaction month is used as the filter key. Records outside this window are excluded from all subsequent stages. This filter is applied before writing the Bronze output so the Bronze layer already contains only in-scope data.

In [0]:
raw_hdb_df = raw_hdb_df.filter(
    (col("month") >= DATE_START) & (col("month") <= DATE_END)
)

print(f"Records after date filter ({DATE_START} to {DATE_END}) : {raw_hdb_df.count():,}")

Records after date filter (2012-01-01 to 2016-12-31) : 92,544


## Write Bronze Output

In [0]:
write_layer(raw_hdb_df,CLD_RAW, "RAW")

RAW written to table : bronze.raw_hdb
